# SFINCS — NJ Sandy: forcing phase-lag viewer

The modeled pre-storm tide peaks **late** vs observations (Sandy Hook +18 min, Shrewsbury +38),
because the northern boundary is interpolated from the harbor-phase **Battery** gauge (Sandy Hook
was excluded — it failed mid-storm). This notebook A/Bs alternative boundary **forcing sources** to
re-phase the coast, keeping the sealed-premier build/waves fixed so only the forcing changes.

The estuary leak/Shark-carve story lives in the archived viewer
`archive/notebooks/sfincs-nj-sandy-viz-estuary-leakfix.ipynb`.

## Setup

In [ ]:
# Viz stack (this import also primes PROJ before hydromt loads).
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from hydromt_sfincs import SfincsModel
from nj_sfincs import plots, validate

EXP_ROOT = ROOT / "experiments"
print("experiments dir:", EXP_ROOT)

## 1. Offshore tidal phase by forcing source — the cheap headline (no SFINCS run)

Each source's series nearest the northern anchor (the Sandy Hook gauge) is cross-correlated
against the **real** Sandy Hook observed tide. **Positive = the source is phase-late at the coast**
and will import a late tide (like the Battery); a source near 0 delivers the observed offshore phase.
Sources whose data file isn't built yet show `n/a`.

Early result (built sources): Battery **+21 min**, Sandy Hook blend **0 min**.

In [ ]:
# label -> data_catalog geodataset key
SOURCES = {
    "NOAA Battery (baseline)": "noaa_sandy_nj",
    "NOAA Sandy Hook blend": "noaa_sandy_nj_shblend",
    "GTSM-ERA5 total": "gtsm_sandy",
    "GTSM tide-only": "gtsm_sandy_tide",
    "FES2014 tide": "fes_sandy_tide",
}
plots.plot_source_phase(SOURCES);

## 2. Modeled gauge phase — Battery vs blend (vs GTSM)

Each run's label carries its pre-storm phase lag as `Δφ +NN min` (+ = model peaks later than obs).
Read the **left** of the "record ends" line (the pre-storm tide) for the interior gauges. Only runs
already on disk are shown.

In [ ]:
# label -> experiment dir under experiments/ (see nj_sfincs.config EXPERIMENTS)
BA = {
    "Battery (baseline)": "phaselag_battery",
    "Sandy Hook blend": "phaselag_shblend",
    "GTSM-ERA5": "phaselag_gtsm",
    "composite": "phaselag_composite",
}
BA = {k: v for k, v in BA.items() if (EXP_ROOT / v / "sfincs_map.nc").exists()}
print("runs present:", BA or "(none yet — run: python run_experiments.py phaselag_battery phaselag_shblend)")
if BA:
    plots.plot_gauge_verification(BA);

## 3. Phase-lag table (minutes; + = model late)

`validate.gauge_phase_lag` per run: Sandy Hook + Shrewsbury from the 10-min his, Shark from the
hourly map at wet channel cells. Success = Sandy Hook shrinks toward 0 with the blend, while the
**interior** Shrewsbury lag barely moves (that is structural conveyance, not forcing).

In [ ]:
rows = {}
for label, name in BA.items():
    d = EXP_ROOT / name
    mod = SfincsModel(str(d), data_libs=[str(ROOT / "data" / "data_catalog.yml")], mode="r")
    validate.read_output(mod)  # loads his + map, no floodmap downscale
    rows[label] = validate.gauge_phase_lag(mod, d)
pd.DataFrame(rows).T if rows else print("no runs yet")

## 4. Regression — did re-phasing hurt the crest or the flood extent?

The blend only re-phases the **pre-storm tide**; Battery still drives the surge crest. Confirm the
HWM residuals and MOTF extent are not degraded vs the Battery baseline (and see GTSM's known ~1 m
crest under-prediction if that arm is present).

In [ ]:
if BA:
    plots.plot_hwm_residual_panels(BA);
    plots.plot_motf_panels(BA);